# Full pipeline - discovery → finalize → agent, end to end

Run all cells. No shell commands. Four ideas keep the result honest:

1. **Feature discovery (LLM)** - real search over which features to try. The model proposes specs; each is scored by tail-disjoint holdout AUC on the **SELECT fold**.
2. **Train/select/test separation** - the search *selects* on one fold (SELECT_FOLD); the winning spec is *reported* on a different fold (TEST_FOLD) it never saw during the search.
3. **Finalize** - deterministic plumbing: full train on the winning spec, report winner **and** baseline on the TEST fold (honest lift), export the demo batch.
4. **Operational agent run** - featurize, classify, recommend, print summary

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

LOCAL = False
BASE_URL   = "http://localhost:8000/v1" if LOCAL else os.getenv("API_URL")
MODEL_NAME = "Qwen/Qwen3-Coder-30B-A3B-Instruct" if LOCAL else os.getenv("API_MODEL_NAME")
API_KEY    = "not-needed" if LOCAL else os.getenv("RIT_API_KEY")

DATA_DIR     = "../data"
CSV          = f"{DATA_DIR}/C28.csv"
FOLDS        = ("0", "1", "2", "3", "4")
SELECT_FOLD  = "1"    
TEST_FOLD    = "0"     
assert SELECT_FOLD != TEST_FOLD

PER_FOLD     = 200    
MAX_ITERS    = 15
PATIENCE     = 5
FEATURE_BUDGET = 150   

SPEC      = f"{DATA_DIR}/best_spec.json"
MODEL     = f"{DATA_DIR}/c28_model.joblib"
METRICS   = f"{DATA_DIR}/c28_metrics.json"
DEMO_DIR  = f"{DATA_DIR}/c28_demo"
DEMO_MAX  = 200
LOG       = f"{DATA_DIR}/discovery_log.jsonl"

WORKDIR   = "../work/agent_run"
TOP_K     = 10
MAX_TURNS = 12


In [ ]:
import sys, json
from pathlib import Path
root = Path().resolve().parent
if str(root) not in sys.path:
    sys.path.append(str(root))

import requests
def chat_fn(system, user):
    """OpenAI-compatible chat call (RIT API or local vLLM). To reuse your
    agent_harness client instead, wrap it to match this signature."""
    r = requests.post(f"{BASE_URL.rstrip('/')}/chat/completions",  #type: ignore
                      headers={"Authorization": f"Bearer {API_KEY}"},
                      json={"model": MODEL_NAME, "temperature": 0.4,
                            "messages": [{"role": "system", "content": system},
                                         {"role": "user", "content": user}]},
                      timeout=180)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

## Phase 1 - Feature discovery (LLM searches; selects on SELECT_FOLD)

Each candidate spec is scored on `SELECT_FOLD`. The best AUC here is the *optimistic* number - it's the max over everything the search tried, so it's biased upward. The honest number comes in Phase 2 on a different fold. The `MAX_FEATURES` cap keeps specs compact (each transform emits several features per channel, so stacking transforms across all ~23 channels balloons fast).

In [ ]:
from scripts.featurize_spec import collect_balanced_sample
from scripts.discovery_agent import make_llm_proposer, run_discovery, StopConfig

sample = collect_balanced_sample(CSV, per_fold=PER_FOLD, folds=FOLDS)
proposer = make_llm_proposer(chat_fn, max_retries=5)

disc = run_discovery(lambda: sample, proposer=proposer, holdout_fold=SELECT_FOLD,  #type: ignore
                     stop=StopConfig(max_iters=MAX_ITERS, patience=PATIENCE,
                                     feature_budget=FEATURE_BUDGET), log_path=LOG)

Path(SPEC).write_text(json.dumps(disc["best_spec"], indent=2))
s = disc["summary"]
print(f"\nSELECT fold {SELECT_FOLD}: baseline {s['baseline_auc']} -> best {s['best_auc']} "
      f"(optimistic / search-time number)")

[discovery] balanced sample: 1000 flights | per-fold counts {'0': 200, '1': 200, '2': 200, '3': 200, '4': 200}
[realdata] fit excluding fold 1: train=800 rows / 61 planes, holdout=200 rows / 16 planes, plane overlap=0, holdout AUC=0.6372030293598921
[discovery] feature cap = 427 (baseline 277 + budget 150)
baseline (summary_stats): AUC=0.6372  [277 features]
[realdata] fit excluding fold 1: train=800 rows / 61 planes, holdout=200 rows / 16 planes, plane overlap=0, holdout AUC=0.6589895217346197
  spec_474e3e9f3b: AUC=0.659 [412 feats, overlap=0, 13.43s]  <-- new best
[realdata] fit excluding fold 1: train=800 rows / 61 planes, holdout=200 rows / 16 planes, plane overlap=0, holdout AUC=0.6551509492685963
  spec_00ba511728: AUC=0.6552 [313 feats, overlap=0, 9.8s]
[realdata] fit excluding fold 1: train=800 rows / 61 planes, holdout=200 rows / 16 planes, plane overlap=0, holdout AUC=0.6360618321402634
  spec_b5203d493a: AUC=0.6361 [394 feats, overlap=0, 34.71s]
[realdata] fit excluding fol

## Phase 2 - Finalize: honest evaluation on TEST_FOLD + plumbing

Trains the winning spec on the full data and reports it on `TEST_FOLD` next to the `summary_stats` baseline on that same fold. Report honest metric lift. If it's ~0, the search overfit the select fold and you should report the baseline. Then export the demo batch from the test fold.

In [4]:
from scripts.featurize_spec import build_training_table
from scripts.realdata import fit_excluding_fold, export_flights_to_dir, NGAFID_NON_SENSOR
import pandas as pd, tempfile, os

spec = json.loads(Path(SPEC).read_text())
m_best = fit_excluding_fold(build_training_table(CSV, spec), TEST_FOLD, MODEL)
Path(METRICS).write_text(json.dumps(m_best, indent=2))

sensors = [c for c in pd.read_csv(CSV, nrows=0).columns if c not in NGAFID_NON_SENSOR]
base_spec = {"spec_id": "baseline_summary_stats",
             "features": [{"transform": "summary_stats", "channels": sensors}]}
m_base = fit_excluding_fold(build_training_table(CSV, base_spec), TEST_FOLD,
                            os.path.join(tempfile.gettempdir(), "baseline.joblib"))

import math
lift = m_best["holdout_roc_auc"] - m_base["holdout_roc_auc"]
n_h = m_best.get("n_holdout", 0)
se = math.sqrt(max(1e-9, m_best["holdout_roc_auc"]*(1-m_best["holdout_roc_auc"]))/max(1, n_h/2))
print(f"TEST fold {TEST_FOLD} (never used to choose the spec; {n_h}-flight holdout, AUC SE ~{se:.3f}):")
print(f"  baseline test-AUC : {m_base['holdout_roc_auc']:.4f} ({m_base['n_features']} feats)")
print(f"  winner   test-AUC : {m_best['holdout_roc_auc']:.4f} ({m_best['n_features']} feats)")
print(f"  HONEST LIFT       : {lift:+.4f}")
if lift <= se:
    print(f"  -> within ~1 SE of zero: no meaningful lift. summary_stats is hard to beat here")
    print(f"     (a clean finding). Rotate the SELECT/TEST pair for a mean +/- spread.")
else:
    print(f"  -> exceeds ~1 SE on the untouched fold; rotate the pair to confirm.")

export_flights_to_dir(CSV, DEMO_DIR, max_flights=DEMO_MAX, only_fold=TEST_FOLD)

  featurized 500 flights
  featurized 1000 flights
  featurized 1500 flights
  featurized 2000 flights
  featurized 2500 flights
  featurized 3000 flights
  featurized 3500 flights
  featurized 4000 flights
  featurized 4500 flights
  featurized 5000 flights
[realdata] fit excluding fold 0: train=3997 rows / 62 planes, holdout=1092 rows / 16 planes, plane overlap=0, holdout AUC=0.7425274520751908
  featurized 500 flights
  featurized 1000 flights
  featurized 1500 flights
  featurized 2000 flights
  featurized 2500 flights
  featurized 3000 flights
  featurized 3500 flights
  featurized 4000 flights
  featurized 4500 flights
  featurized 5000 flights
[realdata] fit excluding fold 0: train=3997 rows / 62 planes, holdout=1092 rows / 16 planes, plane overlap=0, holdout AUC=0.75087559007157
TEST fold 0 (never used to choose the spec; 1092-flight holdout, AUC SE ~0.019):
  baseline test-AUC : 0.7509 (277 feats)
  winner   test-AUC : 0.7425 (412 feats)
  HONEST LIFT       : -0.0083
  -> with

(PosixPath('../data/c28_demo/flights'),
 PosixPath('../data/c28_demo/metadata.csv'))

## Phase 3 - Operational agent run (deterministic chain + LLM summary)

In [5]:
os.environ["DISCOVERY_SPEC"] = str(Path(SPEC).resolve())

Path(WORKDIR).mkdir(parents=True, exist_ok=True)
FEATS = str(Path(WORKDIR) / "feats.csv"); PREDS = str(Path(WORKDIR) / "preds.json")
QUEUE = str(Path(WORKDIR) / "maintenance_queue.jsonl")
FLIGHT_DIR = f"{DEMO_DIR}/flights"; METADATA = f"{DEMO_DIR}/metadata.csv"

if LOCAL:
    from scripts.single_agent.agent_harness import run_agent, order_diff, VLLMClient
    client = VLLMClient(base_url=BASE_URL, model=MODEL_NAME, api_key=API_KEY) #type: ignore
else:
    from scripts.single_agent.agent_harness import run_agent, order_diff, OllamaClient
    client = OllamaClient(base_url=BASE_URL, model=MODEL_NAME, api_key=API_KEY)  #type: ignore

brief_path = Path("../openclaw/agent_task_brief.md")
SYSTEM_PROMPT = brief_path.read_text() if brief_path.exists() else (
    "You are the Orchestrator for a predictive-maintenance workflow. Call tools in "
    "order: inspect, featurize, classify, recommend. Stop after recommend with a HITL summary.")

USER_GOAL = (
    f"Flight directory: {FLIGHT_DIR}\nMetadata: {METADATA}\nModel: {MODEL}\n"
    f"Feature output: {FEATS}\nPredictions output: {PREDS}\nQueue: {QUEUE}\n"
    f"Top-k: {TOP_K}\nRun the workflow and produce the HITL summary.\n"
    f"Model holdout AUC (test fold {TEST_FOLD}, never seen in training): "
    f"{json.load(open(METRICS))['holdout_roc_auc']:.3f}")

run = run_agent(client, SYSTEM_PROMPT, USER_GOAL, max_turns=MAX_TURNS, verbose=True)
print("\n=== HITL SUMMARY ===\n", run.final_summary)
print("\n=== call order ===\n", json.dumps(order_diff(run), indent=2))

[*] Warming up qwen3-coder:30b on the server (this may take a minute)...
[turn 1] inspect({'flight_dir': '../data/c28_demo/flights', 'metadata': '../data/c28_demo/metadata.csv'}) -> OK: {'tool': 'inspect_data', 'n_flights': 200, 'label_balance': {0: 107, 1: 93}, 'n_planes': 16, 'detected_channels': ['volt1', 'volt2', 'amp1', 'amp2', 'FQtyL', 'FQtyR', 'E1 FFlow', 'E1 OilT', 'E1 OilP', 'E1 RPM', 'E1 CHT1', 'E1 CHT2', 'E1 CHT3', 'E1 CHT4', 'E1 EGT1', 'E1 EGT2', 'E1 EGT3', 'E1 EGT4', 'OAT', 'IAS', 'VSpd', 'NormAc', 'AltMSL'], 'n_channels': 23, 'example_flight_steps': 6363, 'notes': 'Read-only. Flags class balance and channel inventory for the planner.'}
[turn 2] featurize({'flight_dir': '../data/c28_demo/flights', 'metadata': '../data/c28_demo/metadata.csv', 'out': '../work/agent_run/feats.csv'}) -> OK: {'tool': 'featurize_flights', 'rows': 200, 'n_features': 412, 'out': '../work/agent_run/feats.csv'}
[turn 3] classify({'feats': '../work/agent_run/feats.csv', 'model': '../data/c28_model.jo